# Get started with `shrecc.NewDatabase`

This is the shortest complete workflow for writing a historical electricity database. Before running it, create a Brightway project containing a basic ecoinvent (3.9 to 3.12) background database.

The output database name is replaced if it already exists. The expanded [analysis notebook](2_shrecc_analysis.ipynb) covers prospective TYNDP databases, multiple years, physical volumes, mapping diagnostics, and other options.

In [ ]:
from shrecc import NewDatabase

## Configure

Edit the project, background database, output name, countries, and year below. With no time selection, SHRECC uses the complete year. A historical year automatically selects Energy Charts as the source.

In [ ]:
# My parameters

years        = 2025 # Either an int or a list
countries    = ["BE", "DE", "FR", "LU"] # A list of countries
project_name = "SHRECCei311" # The name of my Brightway project, where to write the electricity database
bg_db_name   = "ecoinvent-3.11-cutoff" # The name of the background database with which electricity inventories are linked
my_db_name   = "shrecc_2025_annual" # The name of my new electricity database

# My NewDatabase object

electricity_database = NewDatabase(
    years=years,
    countries=countries,
    project_name=project_name,
    bg_db_name=bg_db_name,
    my_db_name=my_db_name,
)

## Create

`create()` obtains or loads the source data, solves the interconnected electricity system, maps activities, and prepares the database table. It does not modify Brightway.

In [ ]:
electricity_database.create()

## Write

`write()` writes the prepared inventory to the configured Brightway project. An existing output database with the same name is replaced.

In [ ]:
electricity_database.write()

In [ ]:
electricity_database.written_database_names

## Check

In [ ]:
import bw2data as bd
import bw2calc as bc
import pandas as pd

In [ ]:
bd.projects.set_current(project_name)

In [ ]:
method = {"impact_categories":[('ecoinvent-3.11',
  'EF v3.1',
  'climate change',
  'global warming potential (GWP100)')]}

demands = {act["location"]:{act["id"]:1} for act in bd.Database(my_db_name)}

data_objs = bd.get_multilca_data_objs(functional_units=demands, method_config=method)

In [ ]:
lca = bc.MultiLCA(
    demands=demands,
    method_config=method,
    data_objs=data_objs
    )

In [ ]:
lca.lci()
lca.lcia()

In [ ]:
# Show results
ax = pd.DataFrame(lca.scores.items()).plot.bar(legend=None)
ax.set_title(f"Annual impacts of 1 kWh from the {years} grid")
ax.set_xticklabels(demands.keys())
ax.set_ylabel(method["impact_categories"][0][3])

## Optional: inspect hourly impacts

SHRECC retains every hourly electricity mix used to build the annual foreground inventory. `lcia()` calculates the impact intensity per kilowatt hour for all hours of the year without writing hourly activities to Brightway.

In [ ]:
hourly_assessment = electricity_database.lcia(
    methods=method["impact_categories"]
)
hourly_assessment

In [ ]:
hourly_intensity = hourly_assessment.hourly(years)["intensity"]
hourly_intensity.sel(
    consumer_country="FR",
    impact_category=hourly_assessment.impact_categories[0],
).plot(marker="o")